In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

In [2]:
SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

Dispositivo: cuda
Torch: 2.11.0+cu128  |  Torchvision: 0.26.0+cu128


In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes

        scale = np.sqrt(2) / (in_channels * out_channels)

        self.weights = nn.Parameter(
            scale * torch.randn(
                in_channels,
                out_channels,
                modes,
                dtype=torch.cfloat
            )
        )

    def forward(self, x):

        batchsize = x.shape[0]

        x_ft = torch.fft.rfft(x)

        out_ft = torch.zeros(
            batchsize,
            self.out_channels,
            x.size(-1)//2 + 1,
            dtype=torch.cfloat,
            device=x.device
        )

        out_ft[:, :, :self.modes] = torch.einsum(
            "bim,iom->bom",
            x_ft[:, :, :self.modes],
            self.weights
        )

        x = torch.fft.irfft(out_ft, n=x.size(-1))

        return x
    
class FourierLayer(nn.Module):
    def __init__(self, width, modes):
        super().__init__()

        self.spectral = SpectralConv1d(width, width, modes)
        self.pointwise = nn.Conv1d(width, width, kernel_size=1)

    def forward(self, x):
        return self.spectral(x) + self.pointwise(x)
    
class FNO1d(nn.Module):
    def __init__(
        self,
        modes=16,
        width=64,
        in_dim=2,
        out_dim=1,
        depth=4
    ):
        super().__init__()

        self.lift = nn.Linear(in_dim, width)

        self.layers = nn.ModuleList([
            FourierLayer(width, modes)
            for _ in range(depth)
        ])

        self.proj1 = nn.Linear(width, 128)
        self.proj2 = nn.Linear(128, out_dim)

    def forward(self, x):

        x = self.lift(x)
        x = x.permute(0, 2, 1)

        for layer in self.layers:
            x = layer(x)
            x = torch.nn.functional.gelu(x)

        x = x.permute(0, 2, 1)

        x = self.proj1(x)
        x = torch.nn.functional.gelu(x)
        x = self.proj2(x)

        return x

batch = 32
n = 256

coords = torch.linspace(0, 1, n)
coords = coords[None, :, None].repeat(batch, 1, 1)

a = torch.randn(batch, n, 1)

inp = torch.cat([a, coords], dim=-1)

model = FNO1d(
    modes=16,
    width=64,
    in_dim=2,
    out_dim=1
)

out = model(inp)

print(out.shape)

torch.Size([32, 256, 1])
